In [ ]:
import json
import geopandas as gpd
import sys
import os
import subprocess
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from shapely.geometry import box, shape, mapping
from shapely.ops import unary_union
import shutil
import importlib

In [ ]:
CONFIG = {
    "hybas_ids": [4090896200, 4090895980],
    "hydrobasins_shp": "/home/jarvis/hybas_level9/hybas_as_lev09_v1c.shp",
    "dem_buffer_deg": 0.08,
    "ridge_accumulation_threshold": 100000,
    "output_dir": Path("./ridge_experiment_output"),
    "grass_mapset": "ridge_experiment",
    "grass_location": "ridge_loc_wgs84",
    "grass_gisdb": "./grassdata",
    "DEM": "FABDEM", # FABDEM/ SRTM1
}

output_dir = CONFIG["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_and_dissolve_watersheds(shp_path, hybas_ids):
    """Load two adjacent watersheds from HydroBASINS and dissolve them."""
    print(f"\n Loading HydroBASINS polygons: {hybas_ids}")
 
    gdf = gpd.read_file(shp_path)
 
    selected = gdf[gdf["HYBAS_ID"].isin(hybas_ids)].copy()
 
    if len(selected) != len(hybas_ids):
        found = selected["HYBAS_ID"].tolist()
        missing = set(hybas_ids) - set(found)
        raise ValueError(f"Could not find HYBAS_IDs: {missing}. Found: {found}")
 
    print(f"    Found {len(selected)} polygons. Dissolving...")
 
    dissolved_geom = unary_union(selected.geometry.values)
    dissolved_gdf = gpd.GeoDataFrame(
        {"geometry": [dissolved_geom], "source": ["dissolved_experiment"]},
        crs=selected.crs
    )
 
    out_path = CONFIG["output_dir"] / "dissolved_watersheds.gpkg"
    dissolved_gdf.to_file(out_path, driver="GPKG")
    print(f"    Dissolved boundary saved: {out_path}")
 
    selected.to_file(CONFIG["output_dir"] / "original_watersheds.gpkg", driver="GPKG")
 
    return dissolved_gdf, selected


In [ ]:
def download_dem_with_buffer(dissolved_gdf):
    """Download SRTM1 DEM tiles covering the dissolved boundary + buffer."""
    print(f"\n Downloading {CONFIG['DEM']} DEM (buffer={CONFIG['dem_buffer_deg']} deg)")
    if (CONFIG["DEM"] == "SRTM1"):
        try:
            import elevation
        except ImportError:
            raise ImportError("pip install elevation  - required for SRTM1 download")
    elif(CONFIG["DEM"] == "FABDEM"):
        try:
            import fabdem
            importlib.reload(fabdem)
        except ImportError:
            raise ImportError("pip install fabdem - required for FABDEM download")
    else:
        print("Specify the DEM to use. Options: SRTM1/ FABDEM")
    bounds = dissolved_gdf.total_bounds  # (minx, miny, maxx, maxy)
    buf = CONFIG["dem_buffer_deg"]
    bbox = (
        bounds[0] - buf,   # west
        bounds[1] - buf,   # south
        bounds[2] + buf,   # east
        bounds[3] + buf,   # north
    )
 
    dem_path = Path(CONFIG["output_dir"]) / "srtm1_buffered.tif"
    dem_path = dem_path.resolve()
    print(f"    Bounding box: {[round(x, 4) for x in bbox]}")

    if (CONFIG["DEM"] == "SRTM1"):
        elevation.clip(
            bounds=bbox,
            output=str(dem_path),
            product="SRTM1",
        )
        elevation.clean()
    elif(CONFIG["DEM"] == "FABDEM"):
        fabdem.download(
            bounds=bbox,
            output_path=str(dem_path),
            show_progress=True,
        )
    else:
        print("Specify the DEM to use. Options: SRTM1/ FABDEM")
    print(f"    DEM saved: {dem_path}")
    return dem_path, bbox

In [ ]:
def build_pseudo_dem_in_grass(dem_path):
    """
    Inside GRASS: import DEM, build pseudo_dem = max_elev - dem.
    Drainage lines on pseudo_dem = ridgelines on real DEM.
    """
    import grass.script as gs
    print("\n Building pseudo-DEM (inversion) in GRASS")

    # Import original DEM
    gs.run_command("r.in.gdal",
        input=dem_path,
        output="dem_original",
        overwrite=True,
        quiet=True
    )
    gs.run_command("g.region", raster="dem_original", flags="p")

    filled_dem_temp = "dem_filled_temp"
    filled_dem = "dem_filled"
    flow_dir = "flow_dir"
    gs.run_command("r.fill.dir", input="dem_original", output=filled_dem_temp, direction=flow_dir, overwrite=True)
    gs.run_command("r.fill.dir", input=filled_dem_temp, output=filled_dem, direction=flow_dir, overwrite=True)
    # Get max elevation value
    
    stats = gs.parse_command("r.univar", map="dem_filled", flags="g")
    max_elev = float(stats["max"])
    print(f"    Max elevation in AOI: {max_elev:.1f} m")
 
    # Build pseudo-DEM: inversion
    # We add 1 to ensure the inversion doesn't produce zeros at the peak cells
    gs.mapcalc(
        f"pseudo_dem = {max_elev + 1} - dem_filled",
        overwrite=True,
        quiet=True
    )
 
    print(f"    pseudo_dem = {max_elev + 1:.1f} - dem_filled")
    print("    (valleys in real DEM -> peaks in pseudo_dem -> will attract 'flow')")
 
    return max_elev

In [ ]:
def run_watershed_on_pseudo_dem():
    """
    r.watershed on pseudo_dem produces:
      - pseudo_drainage   -> flow direction on inverted surface (= uphill on real)
      - pseudo_accumulation -> accumulation on inverted surface (= ridge accumulation)
    """
    print("\n Running r.watershed on pseudo-DEM")
    import grass.script as gs
    # No mask here — we want ridges to form across the full buffered DEM
    # so that ridges at the watershed boundary are fully captured
    gs.run_command("r.watershed",
        elevation="pseudo_dem",
        drainage="pseudo_drainage",
        accumulation="pseudo_accumulation",
        flags="as",          # single-flow direction (D8), consistent with real pipeline
        overwrite=True,
        quiet=True
    )
 
    print("    pseudo_drainage and pseudo_accumulation computed.")


In [ ]:
def extract_ridgelines():
    """
    Cells with high pseudo-accumulation = many cells drain 'uphill' through them
    = they are on a ridge.
 
    This mirrors exactly how you extract drainage lines from real accumulation,
    just on the inverted surface.
    """
    print(f"\n Extracting ridgelines (threshold={CONFIG['ridge_accumulation_threshold']})")

 
    threshold = CONFIG["ridge_accumulation_threshold"]
    import grass.script as gs
    # Binary raster: 1 where pseudo_accum >= threshold
    gs.mapcalc(
        f"ridgeline_raster = if(abs(pseudo_accumulation) >= {threshold}, 1, null())",
        overwrite=True,
        quiet=True
    )
 
    # Thin to 1-pixel skeleton (important for clean vector conversion)
    gs.run_command("r.thin",
        input="ridgeline_raster",
        output="ridgeline_thinned",
        overwrite=True,
        quiet=True
    )
 
    # Vectorize
    gs.run_command("r.to.vect",
        input="ridgeline_thinned",
        output="ridgelines_vec",
        type="line",
        overwrite=True,
        quiet=True,
        flags="s"   # smooth corners
    )
 
    # Export to GeoPackage
    out_path = str(CONFIG["output_dir"] / "ridgelines.gpkg")
    gs.run_command("v.out.ogr",
        input="ridgelines_vec",
        output=out_path,
        format="GPKG",
        overwrite=True,
        quiet=True
    )
 
    print(f"    Ridgelines exported: {out_path}")
    return out_path

In [ ]:
def export_pseudo_accumulation():
    """Export the pseudo-accumulation raster as GeoTIFF for external inspection."""
    print("\n Exporting pseudo-accumulation raster")
    import grass.script as gs
    
    out_path = Path(CONFIG["output_dir"]) / "pseudo_accumulation.tif"
    out_path = str(out_path.resolve())
    gs.run_command("r.out.gdal",
        input="pseudo_accumulation",
        output=out_path,
        format="GTiff",
        type="Float64",
        overwrite=True,
        quiet=True
    )
    print(f"    pseudo_accumulation.tif saved: {out_path}")
    return out_path


In [ ]:
def visualize_comparison(ridgelines_path, original_gdf, dissolved_gdf, dem_path):
    """
    Side-by-side comparison:
    Left:  ridgelines overlaid on DEM hillshade
    Right: ridgelines vs original HydroBASINS boundaries (the alignment test)
    """
    print("\n Generating visual comparison")
 
    import rasterio
    from rasterio.plot import show as rioshow
    from rasterio.warp import reproject, Resampling
 
    # Load ridgelines (or use placeholder if dry-run)
    if ridgelines_path and Path(ridgelines_path).exists():
        ridges_gdf = gpd.read_file(ridgelines_path)
    else:
        print("    No ridgelines file found — generating placeholder plot")
        ridges_gdf = None
 
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="#1a1a2e")
    fig.suptitle(
        "Ridge Extraction Experiment - Alignment with HydroBASINS Boundaries",
        color="white", fontsize=13, y=0.98
    )
 
    for ax in axes:
        ax.set_facecolor("#0f0f1a")
        ax.tick_params(colors='#888888', labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')
 
    ax_left = axes[0]
    ax_left.set_title("Pseudo-accumulation ridgelines on DEM", color="#ccccee", fontsize=10)
 
    with rasterio.open(dem_path) as src:
        dem_data = src.read(1).astype(float)
        dem_data[dem_data == src.nodata] = np.nan
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
 
        # Simple hillshade approximation for visual quality
        from numpy import gradient, arctan, sqrt, pi
        dy, dx = gradient(np.nan_to_num(dem_data))
        slope = arctan(sqrt(dx**2 + dy**2))
        # Light from NW
        aspect = arctan(dy, dx)
        hillshade = (
            np.cos(np.radians(45)) * np.cos(slope) +
            np.sin(np.radians(45)) * np.sin(slope) * np.cos(np.radians(315) - aspect)
        )
        ax_left.imshow(
            hillshade, cmap="gray", extent=extent,
            vmin=-0.5, vmax=1.0, alpha=0.7, aspect='auto'
        )
        ax_left.imshow(
            dem_data, cmap="terrain", extent=extent,
            alpha=0.5, aspect='auto'
        )
 
    # Overlay dissolved boundary
    dissolved_gdf.boundary.plot(ax=ax_left, color="#ffaa00", linewidth=1.5,
                                 linestyle="--", label="Dissolved boundary (input)")
 
    if ridges_gdf is not None and len(ridges_gdf) > 0:
        ridges_gdf.plot(ax=ax_left, color="#00ffcc", linewidth=0.6, alpha=0.8,
                        label="Extracted ridgelines")
 
    ax_left.legend(loc="lower left", fontsize=7,
                   facecolor="#1a1a2e", edgecolor="#333355", labelcolor="white")
 
    ax_right = axes[1]
    ax_right.set_title("Alignment: ridgelines vs original HydroBASINS boundaries", 
                        color="#ccccee", fontsize=10)
 
    # DEM background (lighter, for contrast)
    with rasterio.open(dem_path) as src:
        dem_data = src.read(1).astype(float)
        dem_data[dem_data == src.nodata] = np.nan
        ax_right.imshow(
            dem_data, cmap="gray_r", extent=extent,
            alpha=0.3, aspect='auto'
        )
 
    # Original two watershed boundaries
    original_gdf.boundary.plot(ax=ax_right, color="#ff6644", linewidth=2.0,
                                linestyle="-", label="Original HydroBASINS boundary")
    # Interior shared boundary (the one we care most about) — dissolve highlights it
    dissolved_gdf.boundary.plot(ax=ax_right, color="#ffaa00", linewidth=1.0,
                                 linestyle="--", alpha=0.5, label="Dissolved outer boundary")
 
    if ridges_gdf is not None and len(ridges_gdf) > 0:
        ridges_gdf.plot(ax=ax_right, color="#00ffcc", linewidth=0.8, alpha=0.9,
                        label="Extracted ridgelines")
 
    ax_right.legend(loc="lower left", fontsize=7,
                    facecolor="#1a1a2e", edgecolor="#333355", labelcolor="white")
 
    plt.tight_layout()
    out_path = CONFIG["output_dir"] / "ridge_alignment_comparison.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"    Comparison plot saved: {out_path}")
    return str(out_path)


In [ ]:
def measure_boundary_offset(ridgelines_path, original_gdf):
    """
    Measure how far the extracted ridgelines are from the original
    HydroBASINS boundaries.
 
    Strategy:
      - For each original boundary segment, find the nearest ridgeline point
      - Report percentile statistics of the offset distance
      - Flag segments with large offsets (>500m) — these need manual review
 
    This will tell us:
      a) Whether the ridge extraction is working (offsets should be small)
      b) How much boundary correction is needed
      c) Whether the offset is systematic (shift) or random (noise)
    """
    print("\n[8] Measuring boundary offset distances")
 
    if not ridgelines_path or not Path(ridgelines_path).exists():
        print("    Skipping offset analysis - no ridgelines file")
        return {}
 
    ridges_gdf = gpd.read_file(ridgelines_path)
    if len(ridges_gdf) == 0:
        print("    Empty ridgelines layer - check accumulation threshold")
        return {}
 
    # Reproject to metric CRS for distance measurement
    # Use UTM zone appropriate for the centroid
    centroid = original_gdf.geometry.centroid.iloc[0]
    utm_crs = f"EPSG:{32644}"  # WGS84 UTM nort
 
    original_utm = original_gdf.to_crs(utm_crs)
    ridges_utm = ridges_gdf.to_crs(utm_crs)
 
    # Extract boundary as densified point set (sample every ~50m for analysis)
    boundary_line = original_utm.geometry.iloc[0].boundary
    if hasattr(boundary_line, 'geoms'):
        # MultiLineString — take the shared interior boundary
        # (the inter-watershed boundary, not the outer edge)
        segments = list(boundary_line.geoms)
    else:
        segments = [boundary_line]
 
    # Sample points along the boundary
    from shapely.geometry import MultiLineString
    all_ridges = unary_union(ridges_utm.geometry.values)
 
    offset_distances = []
    sample_interval_m = 50  # measure every 50m along boundary
 
    for seg in segments:
        total_len = seg.length
        num_samples = max(10, int(total_len / sample_interval_m))
        for i in range(num_samples + 1):
            pt = seg.interpolate(i / num_samples, normalized=True)
            dist = all_ridges.distance(pt)
            offset_distances.append(dist)
 
    offsets = np.array(offset_distances)
 
    stats = {
        "n_samples": len(offsets),
        "mean_offset_m": float(np.mean(offsets)),
        "median_offset_m": float(np.median(offsets)),
        "p75_offset_m": float(np.percentile(offsets, 75)),
        "p90_offset_m": float(np.percentile(offsets, 90)),
        "p95_offset_m": float(np.percentile(offsets, 95)),
        "max_offset_m": float(np.max(offsets)),
        "pct_within_90m": float(np.mean(offsets < 90) * 100),   # within 1 SRTM3 cell
        "pct_within_250m": float(np.mean(offsets < 250) * 100),
        "pct_within_500m": float(np.mean(offsets < 500) * 100),
    }
 
    print(f"\n    Offset statistics (boundary pts n={stats['n_samples']}):")
    print(f"      Mean offset:   {stats['mean_offset_m']:.1f} m")
    print(f"      Median offset: {stats['median_offset_m']:.1f} m")
    print(f"      P90 offset:    {stats['p90_offset_m']:.1f} m")
    print(f"      Max offset:    {stats['max_offset_m']:.1f} m")
    print(f"      Within 90m:    {stats['pct_within_90m']:.1f}%")
    print(f"      Within 250m:   {stats['pct_within_250m']:.1f}%")
    print(f"      Within 500m:   {stats['pct_within_500m']:.1f}%")
 
    # Save stats to JSON
    stats_path = CONFIG["output_dir"] / "offset_stats.json"
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"\n    Stats saved: {stats_path}")
 
    # Histogram of offsets
    fig, ax = plt.subplots(figsize=(8, 4), facecolor="#1a1a2e")
    ax.set_facecolor("#0f0f1a")
    ax.hist(offsets, bins=60, color="#00ffcc", alpha=0.8, edgecolor="none")
    ax.axvline(90,  color="#ffaa00", linestyle="--", linewidth=1, label="90m (1 SRTM3 cell)")
    ax.axvline(250, color="#ff6644", linestyle="--", linewidth=1, label="250m")
    ax.set_xlabel("Offset distance (m)", color="#ccccee", fontsize=10)
    ax.set_ylabel("Count", color="#ccccee", fontsize=10)
    ax.set_title("Distribution of ridge-to-boundary offset distances", 
                 color="white", fontsize=11)
    ax.tick_params(colors="#888888")
    ax.legend(facecolor="#1a1a2e", edgecolor="#333355", labelcolor="white", fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor("#333355")
 
    hist_path = CONFIG["output_dir"] / "offset_histogram.png"
    plt.tight_layout()
    plt.savefig(hist_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"    Offset histogram saved: {hist_path}")
 
    return stats

In [ ]:
def setup_grass_session(grassdb: str, epsg: int, project_name: str = "hydro_project"):
    grassdb_path = Path(grassdb).resolve()
    location = project_name
    mapset = "PERMANENT"

    grass_bin = shutil.which("grass")
    if grass_bin is None:
        sys.exit("ERROR: GRASS GIS not found on PATH.")

    result = subprocess.run(
        [grass_bin, "--config", "python_path"],
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONWARNINGS": "ignore"})

    grass_python = result.stdout.strip()
    if grass_python and grass_python not in sys.path:
        sys.path.insert(0, grass_python)
        print(f"[INFO] Added GRASS python path: {grass_python}")

    loc_path = grassdb_path / location
    if not loc_path.exists():
        grassdb_path.mkdir(parents=True, exist_ok=True)
        subprocess.run([grass_bin, "-c", f"EPSG:{epsg}", "-e", str(loc_path)],
                       check=True)
        print(f"[INFO] GRASS location created: {loc_path}")

    # Trying grass_session first, fall back to gsetup
    try:
        from grass_session import Session
        session = Session()
        session.open(gisdb=str(grassdb_path), location=location, mapset=mapset)
        print("grass_session initialised successfully.")
        return session
    except ImportError:
        pass

    # Fallback: manual gsetup
    import grass.script.setup as gsetup
    gsetup.init(str(grassdb_path), location, mapset)
    print("GRASS environment initialised via grass.script.setup")
    return None


In [ ]:
def main():
    print("=" * 65)
    print("  Ridge Extraction Experiment  ")
    print("=" * 65)
 
    # Check HydroBASINS file exists
    shp_path = CONFIG["hydrobasins_shp"]
    if not Path(shp_path).exists():
        print(f"\n[ERROR] HydroBASINS shapefile not found: {shp_path}")
        print("  Update CONFIG['hydrobasins_shp'] to your local path.")
        print("  If you don't have it downloaded, get it from:")
        print("  https://www.hydrosheds.org/products/hydrobasins")
        sys.exit(1)
 
    # Step 1: Load and dissolve watersheds
    dissolved_gdf, original_gdf = load_and_dissolve_watersheds(shp_path, CONFIG["hybas_ids"])
 
    # Step 2: Download DEM
    dem_path, bbox = download_dem_with_buffer(dissolved_gdf)
 
    # Step 3–6: GRASS processing
    setup_grass_session(CONFIG['grass_gisdb'], 32644, CONFIG['grass_location'])

    import grass.script as gs
    
    build_pseudo_dem_in_grass(dem_path)
    run_watershed_on_pseudo_dem()
    ridgelines_path = extract_ridgelines()
    export_pseudo_accumulation()
 
    # Step 7: Visualize
    visualize_comparison(ridgelines_path, original_gdf, dissolved_gdf, dem_path)
 
    # Step 8: Measure offset
    stats = measure_boundary_offset(ridgelines_path, original_gdf)
 
    # ── Interpret results ────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  INTERPRETATION GUIDE")
    print("=" * 65)
 
    if stats:
        median = stats.get("median_offset_m", 999)
        p90    = stats.get("p90_offset_m", 999)
        within_90 = stats.get("pct_within_90m", 0)
 
        if median < 90:
            print(f"\n  [GOOD] Median offset {median:.0f}m < 90m (1 SRTM3 cell).")
            print("  Ridge extraction is producing boundaries close to HydroBASINS.")
            print("  Next step: snap HydroBASINS boundaries to nearest ridgeline.")
        elif median < 300:
            print(f"\n  [ACCEPTABLE] Median offset {median:.0f}m. Ridges are nearby but not tight.")
            print("  Likely cause: HydroBASINS simplification > 1 pixel shift.")
            print("  Next step: increase accumulation threshold to get cleaner ridges,")
            print("  then snap. Check if offset is systematic (1 direction = DEM shift)")
            print("  or random (smoothing artifact).")
        else:
            print(f"\n  [INVESTIGATE] Median offset {median:.0f}m is large.")
            print("  Possible causes:")
            print("    a) Accumulation threshold too low — ridges are noisy")
            print("    b) Flat terrain — ridges are ill-defined (e.g. floodplains)")
            print("    c) HydroBASINS boundary is genuinely not on a ridge here")
            print("       (coastal zones, endorheic basins, flat plateaux)")
            print("  Try: raise CONFIG['ridge_accumulation_threshold'] to 200-500")
            print("  and check the comparison plot for visual confirmation.")
 
    print(f"\n  Output files: {CONFIG['output_dir']}/")
    print("    ridge_alignment_comparison.png  — visual alignment check")
    print("    offset_histogram.png            — offset distance distribution")
    print("    offset_stats.json               — quantitative stats")
    print("    ridgelines.gpkg                 — extracted ridgeline vectors")
    print("    pseudo_accumulation.tif         — raw pseudo-accumulation raster")
    print("=" * 65)
 
 
if __name__ == "__main__":
    main()
 
